# Tutorial 13 — Changing the prospection start year

By default an AeroMAPS scenario calibrates on **historic data over 2000–2019** and
runs its **prospective (forward-looking) trajectory from 2020**. This tutorial shows
**everything you must change to move that boundary** — here, to start the prospection
in **2025** (historic data 2000–2024).

We keep the other temporal bounds fixed (the supported scope of this feature):

| bound | value | role |
|---|---|---|
| `historic_start_year` | 2000 | first year of aviation historic data |
| `prospection_start_year` | **2020 → 2025** | first projected year (what we change) |
| `end_year` | 2050 | last year of the horizon |
| `climate_historic_start_year` | 1940 | first year of the climate (temperature) history |

> **One mental model to remember.** Internally AeroMAPS anchors every calibration on
> `last_historical_year = prospection_start_year - 1`. With the default it is **2019**;
> after this change it becomes **2024**. Most `*_2019`-style parameters were renamed to
> `*_last_historical_year` precisely so they follow this anchor automatically.

## What you must provide — overview

Moving the prospection start year touches **three kinds of inputs**:

1. **Extended historic vectors** (`*_init`) — they must cover *exactly* the historic
   window `[historic_start_year, prospection_start_year-1]`. For a 2025 start that is
   **2000–2024 = 25 values** (instead of 20).
2. **`*_last_historical_year` reference scalars** — values measured *at* the last
   historic year (now 2024): market shares, energy shares, and (for the elasticity
   demand models) `gdp_per_capita_last_historical_year`.
3. **Fixed-reference-year parameters that DO *not* move** — the CORSIA and
   carbon-budget baselines are tied to regulatory/scientific years (2019), independent
   of where your prospection starts.

Plus two behavioural consequences (no action needed, but good to understand):
the **COVID model becomes inert** when the whole COVID period is now historic, and a
few **reference-year trajectories** emit a harmless warning.

We go through each below.

## Step 1 — Set `prospection_start_year` and extend the historic vectors

The new prospection year and the extended historic vectors live in a user inputs JSON
referenced from `config.yaml` as `data.inputs.json_inputs_file`. Let's look at the one
shipped with this tutorial (`data/inputs_2025.json`).

In [ ]:
import json

inp = json.load(open("data/inputs_2025.json"))

# Year bounds travel WITH the vectors (see the note below)
print({k: inp[k] for k in ["historic_start_year", "prospection_start_year", "end_year"]})

# Every *_init vector now has 25 values: one per historic year 2000..2024
for k, v in inp.items():
    if k.endswith("_init"):
        print(f"{k:32s} {len(v)} values   (2000={v[0]:,.0f}  2019={v[19]:,.0f}  2024={v[24]:,.0f})")

**The seven historic vectors** (`rpk_init`, `ask_init`, `rtk_init`, `pax_init`,
`freight_init`, `energy_consumption_init`, `total_aircraft_distance_init`) must each
carry the real observed values up to 2024 — **including the COVID dip in 2020 and the
recovery through 2024**. *(The numbers in this tutorial are illustrative — replace them
with your real historic data.)*

Two rules the engine enforces:

* **Length must match exactly.** `len(vector) == prospection_start_year - historic_start_year`
  (here `2025 - 2000 = 25`). A mismatch raises a clear error (demonstrated next).
* **Put the year bounds in the *same* file as the vectors.** `historic_start_year` and
  `prospection_start_year` define the index the vectors are aligned to, so an inputs file
  that supplies `*_init` vectors must also declare those two years.

In [ ]:
# What happens if a vector has the wrong length? A clear, actionable error.
from aeromaps.utils.functions import _dict_from_parameters_dict

bad = {
    "historic_start_year": 2000,
    "prospection_start_year": 2025,
    "rpk_init": inp["rpk_init"][:20],
}  # only 20 values -> too short for a 2025 start
try:
    _dict_from_parameters_dict(bad)
except ValueError as e:
    print("ValueError:", e)

## Step 2 — Update the `*_last_historical_year` reference scalars

These parameters are **values at the last historic year** (now 2024). They were renamed
from the old `*_2019` names so they track `prospection_start_year` automatically — but the
**numbers** are still your responsibility to update to the new anchor year.

Most live in `markets.yaml` (this tutorial uses the default markets file, so they keep
their illustrative 2019 values — update them to 2024 actuals for a real study):

```yaml
short_range:
  inputs:
    initial:
      rpk_share_last_historical_year: 27.2      # share of total RPK in 2024 [%]
      energy_share_last_historical_year: 26.01  # share of total energy in 2024 [%]
freight:
  inputs:
    initial:
      rtk_share_last_historical_year: 100.0
      energy_share_last_historical_year: 15.0
```

If you use an **elasticity demand model** (`global.demand.model: constant_elasticity`
or `logistic_income`) you additionally provide, at the new anchor year:
`gdp_per_capita_last_historical_year`, and 25-element `gdp_per_capita_init` /
`population_init` vectors.

## Step 3 — Parameters that stay anchored to a *fixed* year (do **not** move them)

Some `…_2019`-looking quantities are **not** "last historical year" — they encode fixed
regulatory or scientific reference years and must stay put when you move the prospection
start. They are exposed as explicit parameters (default **2019**):

| parameter | default | meaning |
|---|---|---|
| `corsia_reference_year` | 2019 | CORSIA offsetting baseline year (regulatory) |
| `carbon_budget_reference_year` | 2019 | year the carbon budget is measured from |
| `world_co2_emissions_carbon_budget_reference_year` | 43.05 | world CO₂ at the budget reference year [GtCO₂] |

Because these reference years (2019) fall **inside your historic window** (2000–2024),
everything resolves correctly without any change. Leave them at 2019 unless you are
deliberately modelling a different regulatory/budget regime.

A companion output, `cumulative_co2_emissions_from_carbon_budget_reference_year`, sums
emissions from `carbon_budget_reference_year` onward so it stays directly comparable to
the aviation carbon budget regardless of where your prospection starts.

## Step 4 — The COVID model becomes inert (automatically)

When the whole COVID episode is now in the **historic** window
(`prospection_start_year > covid_end_year`, here 2025 > 2024), AeroMAPS **skips** every
COVID modelling step so it cannot double-count the dip that is already in your historic
data:

* `covid_load_factor_2020`, the per-market COVID recovery interpolation, and the energy
  COVID corrections are **not applied**;
* the elasticity demand models set their `covid_shift` to **0**.

So the observed 2020 drop must be present in your `*_init` data — which is exactly what
Step 1 provides. We verify this after running (the 2020 load factor stays at the modelled
value instead of being forced to the COVID override of 65.2 %).

## Step 5 — Reference-year trajectories you may want to revisit (non-fatal warnings)

A few interpolated trajectories declare reference years starting at **2020** (e.g. energy
mean-fuel-selling-price curves and `exogenous_carbon_price_reference_years`). With a 2025
start you will see warnings like:

```
The first reference year (2020) differs from the prospection start year (2025).
Interpolation will begin at the first reference year.
```

These are **harmless** — interpolation simply starts at the first reference year — but for
a clean study you may want to update those `*_reference_years` lists to begin at (or after)
your new prospection year.

## Step 6 — Run the scenario

In [ ]:
import warnings
from aeromaps import create_process

warnings.simplefilter("ignore")  # silence the Step-5 reference-year notices for readability

process = create_process(configuration_file="data/config.yaml")
process.compute()

print("prospection_start_year :", process.parameters.prospection_start_year)
print("last_historical_year   :", process.parameters.last_historical_year)

In [ ]:
import pandas as pd

vo = process.data["vector_outputs"]
df = vo if isinstance(vo, pd.DataFrame) else pd.DataFrame(vo)
df.index = range(2000, 2000 + len(df))

years = [2019, 2020, 2024, 2025, 2050]
table = pd.DataFrame(
    {
        "RPK [Gpax-km]": [df.loc[y, "rpk"] / 1e9 for y in years],
        "load_factor [%]": [df.loc[y, "load_factor"] for y in years],
    },
    index=years,
).round(1)
print(table)

print()
print(
    "COVID dip preserved in historic data? rpk[2020] =",
    round(df.loc[2020, "rpk"] / 1e9, 1),
    "Gpax-km  (deep dip, from your *_init, NOT the recovery model)",
)
print(
    "covid_load_factor_2020 (65.2) NOT applied? load_factor[2020] =",
    round(df.loc[2020, "load_factor"], 1),
    "%  (stays at the modelled value)",
)

## Common errors and how to read them

**Stale `…_2019` config key.** Old parameter names are not silently accepted — you get a
self-documenting error telling you the new name. (No backward-compatible alias is provided
on purpose, so wrong numbers can never slip through unnoticed.)

In [ ]:
# Example: an old-style market share key that was renamed
process_demo = create_process(configuration_file="data/config.yaml")
process_demo.parameters.short_range_rpk_share_2019 = 27.2  # stale name a user might keep
try:
    process_demo._check_renamed_inputs()
except ValueError as e:
    print("ValueError:", e)

## Summary — the checklist

To move `prospection_start_year` to year *Y*:

1. **`prospection_start_year: Y`** in your inputs JSON, alongside `historic_start_year`
   (and `end_year`).
2. **Extend every `*_init` vector** to exactly `Y - historic_start_year` values, with real
   observed data through `Y-1` (including the COVID years if they are now historic).
3. **Update the `*_last_historical_year` scalars** (market/energy shares, and
   `gdp_per_capita_last_historical_year` for elasticity demand) to their `Y-1` values.
4. **Leave the fixed reference years alone** (`corsia_reference_year`,
   `carbon_budget_reference_year`) unless you really mean to change the regime.
5. *(Optional)* update any `*_reference_years` trajectories that still start at 2020.

What you do **not** need to touch: `historic_start_year`, `end_year`,
`climate_historic_start_year`, the climate temperature history CSV (1940–1999), or the
COVID parameters (they become inert automatically when the COVID period is historic).

In [ ]:
process.plot("air_transport_co2_emissions")